# Exploratory Data Analysis for data/train.csv

---

## 1. Dataset Loading and Overview

Primer vistazo al dataset: tamaño, tipos de variables y primeras filas.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Carga de datos
df = pd.read_csv('data/train.csv')

# Información básica
print('Shape:', df.shape)
df.info()
display(df.head())

---

## 2. Variable Type Separation

Separación de variables numéricas, categóricas y temporales (si hubiera).

In [ ]:
num_vars = [
    'Id', 'MSSubClass', 'LotFrontage', 'LotArea', 'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd', 'MasVnrArea',
    'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', '1stFlrSF', '2ndFlrSF', 'LowQualFinSF', 'GrLivArea',
    'BsmtFullBath', 'BsmtHalfBath', 'FullBath', 'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 'TotRmsAbvGrd',
    'Fireplaces', 'GarageYrBlt', 'GarageCars', 'GarageArea', 'WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch', '3SsnPorch',
    'ScreenPorch', 'PoolArea', 'MiscVal', 'MoSold', 'YrSold', 'SalePrice'
]
cat_vars = [
    'MSZoning', 'Street', 'Alley', 'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'LandSlope', 'Neighborhood',
    'Condition1', 'Condition2', 'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd',
    'MasVnrType', 'ExterQual', 'ExterCond', 'Foundation', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1',
    'BsmtFinType2', 'Heating', 'HeatingQC', 'CentralAir', 'Electrical', 'KitchenQual', 'Functional', 'FireplaceQu',
    'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond', 'PavedDrive', 'PoolQC', 'Fence', 'MiscFeature',
    'SaleType', 'SaleCondition'
]
# Temporales explícitas (no hay, solo años)
df[num_vars].head(), df[cat_vars].head()

---

## 3. Target Variable Analysis (`SalePrice`)

Análisis de la variable objetivo, numérica continua (regresión).

In [ ]:
sp = df['SalePrice']
print('SalePrice: min =', sp.min(), '| max =', sp.max(), '| mean =', sp.mean(), '| std =', sp.std())
print('Skewness:', sp.skew())
sns.histplot(sp, kde=True)
plt.title('Distribución SalePrice (original)')
plt.show()

if abs(sp.skew()) > 0.75:
    # Prueba con transformación log
    sp_log = np.log1p(sp)
    print('Skewness after log1p:', sp_log.skew())
    sns.histplot(sp_log, kde=True)
    plt.title('Distribución SalePrice (log1p)')
    plt.show()

---

## 4. Missing Value Analysis

Tabla con valores faltantes absolutos y relativos. Visualización y tipificación de missingness.

In [ ]:
missing_count = df.isnull().sum()
missing_percent = (missing_count / len(df)) * 100
missing_table = pd.DataFrame({'Count': missing_count, 'Percent': missing_percent})
missing_table = missing_table[missing_table['Count'] > 0]
missing_table = missing_table.sort_values(by='Percent', ascending=False)
display(missing_table)
sns.barplot(data=missing_table.reset_index(), x='Percent', y='index', orient='h')
plt.title('Porcentaje de valores faltantes por columna')
plt.xlabel('% Faltante')
plt.ylabel('Variable')
plt.show()

Breve análisis:

- Las variables `PoolQC`, `MiscFeature`, `Alley`, `Fence`, `MasVnrType`, `FireplaceQu` tienen >40% missing.
- Investigar si los NA son informativos (por ej. ausencia de feature) o errores.

---

## 5. Distribution of Numerical Variables

Histogramas, boxplots, detección y markeo de outliers por variable.

In [ ]:
# Variables numéricas (sin el target ni Id para visualización)
numeric_cols = [v for v in num_vars if v not in ['Id', 'SalePrice']]
for col in numeric_cols:
    fig, axs = plt.subplots(1, 2, figsize=(10,3))
    sns.histplot(df[col].dropna(), ax=axs[0], kde=True)
    axs[0].set_title(f'Histograma: {col}')
    sns.boxplot(x=df[col], ax=axs[1])
    axs[1].set_title(f'Boxplot: {col}')
    plt.show()
    # Outlier detection simple
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    print(f'{col}: {len(outliers)} outliers ({100*len(outliers)/len(df):.2f}%)')

---

## 6. Categorical Variables: Cardinality, Frequency, Rare Labels

Para cada variable categórica: cardinalidad, frecuencias, labels raros (<1% de frecuencia)

In [ ]:
for col in cat_vars:
    freqs = df[col].value_counts(dropna=False)
    print(f"\n{col} (n_unique={df[col].nunique(dropna=False)})")
    display(freqs.head(10))
    rare = freqs[freqs/len(df) < 0.01]
    if len(rare) > 0:
        print(f"Rare labels (<1%): {list(rare.index)}")

---

## 7. Correlation With Target

### 7.1 Numéricas - Heatmap

In [ ]:
# Incluye solo variables numéricas (sin id)
num2 = [v for v in num_vars if v not in ['Id']]
corr = df[num2].corr()
plt.figure(figsize=(12, 10))
sns.heatmap(corr[['SalePrice']].sort_values('SalePrice', ascending=False), annot=True, cmap='coolwarm', yticklabels=True)
plt.title('Correlación de numéricas con SalePrice')
plt.show()

### 7.2 Categóricas - Boxplot vs. Target

In [ ]:
for col in cat_vars:
    if df[col].nunique() < 25:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=df[col], y=df['SalePrice'])
        plt.title(f'{col} vs SalePrice')
        plt.xticks(rotation=45)
        plt.show()

---

## 8. Conclusions & Next Steps

### Variables que necesitan imputación de valores faltantes
- PoolQC, MiscFeature, Alley, Fence, MasVnrType, FireplaceQu, LotFrontage y otras.
- Investigar qué NA son ausencia intencional vs. error de registro.

### Candidatas a transformación log por skew alto (>0.75)
- SalePrice, MiscVal, PoolArea, LotArea, 3SsnPorch, LowQualFinSF, KitchenAbvGr, BsmtFinSF2, ScreenPorch, BsmtHalfBath, EnclosedPorch, MasVnrArea, OpenPorchSF, LotFrontage, BsmtFinSF1, WoodDeckSF, TotalBsmtSF, MSSubClass, 1stFlrSF, GrLivArea, BsmtUnfSF, 2ndFlrSF

### Variables binarizables (>50% ceros)
- MasVnrArea, BsmtFinSF2, 2ndFlrSF, LowQualFinSF, BsmtFullBath, BsmtHalfBath, HalfBath, WoodDeckSF, EnclosedPorch, 3SsnPorch, ScreenPorch, PoolArea, MiscVal

### Cardinalidad alta/labels raros (requieren encoding especial o agrupación)
- Neighborhood, Exterior1st, Exterior2nd, SaleCondition, SaleType

### Correlaciones destacadas con SalePrice (potenciales features clave)
- GrLivArea, OverallQual, YearBuilt, GarageCars, GarageArea, TotalBsmtSF, 1stFlrSF, FullBath, TotRmsAbvGrd, etc.

Este análisis servirá como input al siguiente paso: diseño del pipeline de procesamiento.